In [15]:
import json
import spacy
from spacy.pipeline import EntityRuler
import inflect
import pandas as pd

p = inflect.engine()

model = spacy.load('en_core_web_sm')

In [16]:
with open('list files\\compounds.json', 'r') as f:
    food_compounds = json.load(f)

with open('list files\\food concentrations.json', 'r') as f:
    food_concentrations = json.load(f)

all_compounds = []

with open('aromatase inhibitors list.json', 'r') as f:
    all_compounds += json.load(f)

with open('phytoestrogens list.json', 'r') as f:
    all_compounds += json.load(f)

with open('phytoprogestogens list.json', 'r') as f:
    all_compounds += json.load(f)

food_set = set()

for compound, foods in food_compounds.items():
    for food in foods:
        food_set.add(food)

for food, compound, low, high in food_concentrations:
    if not pd.isna(food):
        food_set.add(food)

properties = ['aromatase', 'estrogen', 'progesterone', 'phytoestrogen', 'phytoprogestogen', 'estrogens', 'phytoestrogens', 'phytoprogestogens']

biospecimens = ['blood', 'urine', 'serum', 'plasma']

In [17]:
ruler = model.add_pipe('entity_ruler', before='ner', config={"phrase_matcher_attr": "LOWER"})

ner = model.get_pipe('ner')

ner.add_label('FOOD')
ner.add_label('COMPOUND')

ruler.add_patterns([{'label': 'FOOD', 'pattern': food} for food in list(food_set)])
ruler.add_patterns([{'label': 'FOOD', 'pattern': p.plural(food).lower()} for food in list(food_set)])
ruler.add_patterns([{'label': 'COMPOUND', 'pattern': [{'lower': compound.lower()}]} for compound in all_compounds])
ruler.add_patterns([{'label': 'PROPERTY', 'pattern': prop} for prop in properties])
ruler.add_patterns([{'label': 'BIOSPECIMEN', 'pattern': bio} for bio in biospecimens])

In [18]:
import os
model.to_disk(os.getcwd() + '\\model')

In [19]:
import os

model = spacy.load(os.getcwd() + '\\model')

In [20]:
print(all_compounds)

['Apigenin', 'Biochanin A', 'Casimoroin', 'Chrysin', 'Conjugated linoleic acid', 'Corynesidone A', 'Coumestrol', 'Daidzein', 'Enterodiol', 'Enterolactone', 'Enterolactone', 'Epigallocatechin gallate', 'Galangin', 'Garcinone D', 'Garcinone E', 'Genistein', 'Gossypetin', 'Indole-3-carbinol', 'Kaempferol', 'Liquiritigenin', 'Luteolin', 'Melatonin', 'Myricetin', 'Nobiletin', 'Pinocembrin', 'Pinostrobin', 'Pomiferin', 'Quercetin', 'Resveratrol', 'Ursolic acid', 'Î±-mangostin', 'Î³-mangostin', '7-Hydroxyflavone', 'Acacetin', 'Anethole', 'Apigenin', 'Astragalin', 'Aurin', 'Baicalein', 'Baicalin', 'Biochanin A', 'Catechin', 'Chrysin', 'Coumestrin', 'Coumestrol', 'Curcumin', 'Cyanidin', 'Daidzein', 'Daidzin', 'Delphinidin', 'Dihydrodaidzein', 'Dihydrogenistein', 'Enterodiol', 'Enterolactone', 'Epicatechin', 'Epicatechin', 'Epigallocatechin', 'Fisetin', 'Formononetin', 'Formononetin', 'Galangin', 'Genistein', 'Genistin', 'Glabrene', 'Glabridin', 'Hesperetin', 'Hesperidin', 'Icaritin', 'Isoliquir

In [23]:
test_sentences = [
    'what compounds are contained in an apple?',
    'what compounds are contained in an Apple?',
    'how many apples are in a granola bar?',
    'how many blackberries are in a serving?',
    'which foods are good sources of Apigenin?',
    'how many Coffees can i drink in a day?',
    'can i eat mangoes every day?',
    'how many soy milks is enough?',
    'which foods are sources of phytoprogestogens?',
    'at which concentrations has genistein been detected in the blood?',
    'which foods contain luteolin?'
]

print('Mulberry' in list(food_set))

print(p.plural('mulberry'))
print(p.singular_noun('soy milks'))
print(p.singular_noun('phytoprogestogens'))
print(p.singular_noun('luteolin'))

for sentence in test_sentences:
    sent_doc = model(sentence)
    print([(ent, ent.label_) for ent in sent_doc.ents])

True
mulberries
soy milk
phytoprogestogen
False
[(apple, 'FOOD')]
[(Apple, 'FOOD')]
[(apples, 'FOOD')]
[]
[(Apigenin, 'COMPOUND')]
[(Coffees, 'FOOD'), (a day, 'DATE')]
[(mangoes, 'FOOD'), (every day, 'DATE')]
[(soy milks, 'FOOD')]
[(phytoprogestogens, 'PROPERTY')]
[(genistein, 'COMPOUND'), (blood, 'BIOSPECIMEN')]
[(luteolin, 'COMPOUND')]
